<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/14_capstone_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[⬅ Back to the course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · **Notebook 14 of the course**


# 🏆 Notebook 14 — The capstone challenge

> Everything you learned today, in one hour, against the clock and against each other.

**The task.** Predict **90-day mortality** for a set of ICU stays the model has never seen.
We have locked away a **held-out test set of ICU stays** (split *by patient*, fixed seed, identical
for everyone in the room). You build whatever you like on the training stays; when you're ready you
call `score()` and your result goes on the leaderboard.

**The rules** — these are the habits of the whole day, made enforceable:

| ✅ Allowed | ❌ Not allowed |
|-----------|---------------|
| any features you can build from the training data | using `died_in_hosp` (that's target leakage — NB08) |
| any model, any library | fitting *anything* on the test stays, including a scaler or imputer |
| cross-validation on the training stays, as much as you like | tuning by repeatedly resubmitting to `score()` until it looks good |
| ensembling, stacking, calibration | mixing blocs of the same stay across the split |

> 🎯 **Target to beat:** the starter baseline below scores about **0.75** ROC-AUC on the held-out
> stays (≈0.78 in cross-validation on the training stays — that gap is itself a lesson). Good
> time-series feature engineering typically buys you **0.01–0.04**. But the held-out set is only
> ~420 patients, so its 95% confidence interval is roughly **±0.05** — the scorer prints it, and you
> should not believe a lead smaller than that. Getting **0.93+ means you leaked** — go find the bug,
> that's the real prize.

**⏱️ Time:** 45–60 min · **Prerequisites:** Notebooks 04, 05, 06 (08 and 09 help a lot).

## ⚙️ Run me first

In [1]:
# === ⚙️  Workshop setup — run this cell first ===============================
# Works in Google Colab and in local Jupyter. Installs anything missing, sets a
# clean plotting style, and gives you helpers to load the data.
# (This cell is identical in every notebook of the course.)
import importlib.util, subprocess, sys, os, random, warnings
warnings.filterwarnings("ignore")

# --- reproducibility: everyone in the room gets the same numbers ---------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

# 📥 Where the workshop data comes from — already set up for you, nothing to do.
# The data downloads automatically the first time you need it. If you were given a
# different link, just paste it in place of the one below. These forms all work:
#   • a Google-Drive folder link      • a Drive / Dropbox / OneDrive file link
#   • a folder URL ending in "/"      • a link straight to a .zip
# Set it to "" if you would rather upload the CSVs by hand.
# (The data is not in the GitHub repo: it is real de-identified patient data covered
#  by a data use agreement and may not be redistributed openly.)
WORKSHOP_DATA_URL = os.environ.get(
    "WORKSHOP_DATA_URL",
    "https://drive.google.com/drive/folders/1y7CparhrqdlCAZq6xQlj8fZda394fniD")

def _ensure(pkgs):
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
_ensure({"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn",
         "matplotlib":"matplotlib","seaborn":"seaborn","shap":"shap","xgboost":"xgboost"})

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(RANDOM_STATE)          # seeds the legacy global np.random.* calls
RNG = np.random.default_rng(RANDOM_STATE)   # the modern generator — use this one
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5); plt.rcParams["figure.dpi"] = 110

# Every model, split and resample in this course passes random_state=RANDOM_STATE, so
# your numbers should match your neighbour's exactly. (Different library *versions* can
# still shift the last decimal — that is normal and not a mistake on your part.)

# --- data loading: works locally AND remembers your upload across notebooks -----
_CACHE = {"dir": "unset"}   # memo so we only touch Google Drive once per session

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _drive_cache():
    """In Google Colab, mount Drive ONCE and return a persistent folder. A file you
    upload in one notebook is saved here, so every other notebook opens it automatically
    — no re-uploading. Returns None outside Colab, or if you decline to connect Drive."""
    if _CACHE["dir"] != "unset":
        return _CACHE["dir"]
    result = None
    if _in_colab():
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            result = "/content/drive/MyDrive/sepsis_workshop_data"
            os.makedirs(result, exist_ok=True)
        except Exception:
            result = None
    _CACHE["dir"] = result
    return result

def _find(name):
    """Look for the file on disk. Deliberately does NOT touch Google Drive, so the normal
    path never triggers an authorisation popup."""
    for p in [name, f"data/{name}", f"../data/{name}", f"workshop/data/{name}"]:
        if os.path.exists(p):
            return p
    return None

def _find_in_drive(name):
    """Only used as a fallback, because it mounts Drive (and that means a popup)."""
    cache = _drive_cache()
    if cache:
        p = os.path.join(cache, name)
        if os.path.exists(p):
            return p
    return None

def _direct_url(u):
    """Turn an ordinary Google-Drive / Dropbox / OneDrive *share* link into one that a
    plain HTTP client can actually download, so you can paste the link you were given."""
    import re
    m = (re.search(r"drive\.google\.com/file/d/([\w-]+)", u)
         or re.search(r"drive\.google\.com/(?:open|uc)\?(?:export=\w+&)?id=([\w-]+)", u))
    if m:
        return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "dropbox.com" in u:
        return u.split("?")[0] + "?dl=1"
    if "sharepoint.com" in u or "1drv.ms" in u:
        return u + ("&" if "?" in u else "?") + "download=1"
    return u

def _fetch(url, dest):
    import urllib.request, shutil as _sh
    req = urllib.request.Request(_direct_url(url), headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        _sh.copyfileobj(r, f)

_FOLDER = {"done": False}

def _gdrive_folder(name):
    """WORKSHOP_DATA_URL points at a Google-Drive *folder*: fetch it once with gdown
    (a folder cannot be downloaded with a plain HTTP request), then serve files from it."""
    dest = "_workshop_data"
    if not _FOLDER["done"]:
        _ensure({"gdown": "gdown"})
        import gdown
        print("⬇  fetching the workshop data from Google Drive (just once) …")
        gdown.download_folder(url=WORKSHOP_DATA_URL, output=dest, quiet=True, use_cookies=False)
        _FOLDER["done"] = True
    for root, _dirs, files in os.walk(dest):
        if name in files:
            return os.path.join(root, name)
    return None

def _try_download(name):
    """Fetch the data from WORKSHOP_DATA_URL, if one was configured."""
    u = (WORKSHOP_DATA_URL or "").strip()
    if not u:
        return None
    try:
        if "/drive/folders/" in u:
            return _gdrive_folder(name)
        if u.lower().split("?")[0].endswith(".zip"):
            import zipfile
            bundle = "_workshop_data.zip"
            if not os.path.exists(bundle):
                print("⬇  downloading the workshop data bundle …")
                _fetch(u, bundle)
            with zipfile.ZipFile(bundle) as z:      # flatten any folder inside the zip
                for member in z.namelist():
                    if os.path.basename(member) == name:
                        with z.open(member) as src, open(name, "wb") as dst:
                            dst.write(src.read())
                        return name
            print(f"  ({name} was not inside the bundle)")
            return None
        print(f"⬇  downloading {name} …")
        _fetch(u.rstrip("/") + "/" + name, name)
        return name
    except Exception as e:
        print(f"  (download failed: {e})")
        for leftover in (name, "_workshop_data.zip"):
            if os.path.exists(leftover) and os.path.getsize(leftover) == 0:
                os.remove(leftover)
        return None

def _cache_to_drive(name, data):
    cache = _drive_cache()
    if cache:
        dest = os.path.join(cache, name)
        data.to_csv(dest, index=False)
        print(f"  💾 saved to Google Drive ({dest}) — no need to fetch it again.")

def load_csv(name):
    """Load a data CSV: looks on disk, then downloads it from WORKSHOP_DATA_URL, then checks
    your Google-Drive cache, and only as a last resort asks you to upload it — in which case
    it saves a copy to Drive so you never have to upload it twice."""
    p = _find(name)                       # 1. already on disk?
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    p = _try_download(name)               # 2. the built-in download link
    if p:
        print(f"✓ loaded {name}")
        return pd.read_csv(p)
    p = _find_in_drive(name)              # 3. a copy you saved on a previous run
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    try:                                  # 4. last resort: upload it by hand
        from google.colab import files
        print(f"⤴  Upload {name} just once — I'll save it so the other notebooks open it automatically:")
        up = files.upload()
        fname = list(up.keys())[0]
        data = pd.read_csv(fname)
        _cache_to_drive(name, data)
        return data
    except Exception:
        raise FileNotFoundError(
            f"Could not find {name}. Either paste your download link into WORKSHOP_DATA_URL at "
            f"the top of this cell, or put the CSV next to this notebook / in a data/ folder."
        )


## 📂 Load the data and lock the test set

We split **by `icustayid`** — never by row — so no patient appears on both sides. The seed is fixed
so everyone in the room gets exactly the same split and the leaderboard is comparable.

In [2]:
def build_patient_table(ts):
    """Aggregate the 4-hourly time-series into ONE row per ICU stay.
       (This is exactly what Notebook 04 teaches you to build.)"""
    df = ts.copy()
    # clean temperature: prefer Celsius; repair obvious Fahrenheit-entry errors; drop impossible
    tf = df["Temp_F"].where((df["Temp_F"] >= 90) & (df["Temp_F"] <= 110))
    tc = df["Temp_C"].where((df["Temp_C"] >= 25) & (df["Temp_C"] <= 45))
    df["Temp_C_clean"] = tc.fillna((tf - 32) * 5/9)
    # impossible vitals -> NaN, then forward/back-fill WITHIN each stay
    for c, lo, hi in [("HR",20,300),("SysBP",40,300),("MeanBP",20,220),("RR",3,80),("SpO2",30,100)]:
        df[c] = df[c].where((df[c] >= lo) & (df[c] <= hi))
    g = df.groupby("icustayid", group_keys=False)
    vit = ["HR","SysBP","MeanBP","RR","SpO2","Temp_C_clean"]
    df[vit] = g[vit].apply(lambda x: x.ffill().bfill())
    # winsorise skewed labs so one stray value can't define a min/max
    for c in ["Arterial_lactate","Creatinine","BUN","WBC_count","Platelets_count","INR","Glucose"]:
        lo, hi = df[c].quantile(0.01), df[c].quantile(0.99)
        df[c] = df[c].clip(lo, hi)
    tv = ["HR","SysBP","MeanBP","RR","SpO2","Temp_C_clean","GCS","Creatinine","BUN",
          "Arterial_lactate","WBC_count","Platelets_count","Potassium","Sodium",
          "Albumin","INR","Arterial_pH","Shock_Index","SOFA","SIRS","PaO2_FiO2","Hb"]
    grp = df.groupby("icustayid")
    f = {}
    f["age"]=grp["age"].first(); f["gender"]=grp["gender"].first()
    f["elixhauser"]=grp["elixhauser"].first(); f["re_admission"]=grp["re_admission"].first().astype(int)
    f["weight_kg"]=grp["Weight_kg"].median(); f["n_blocs"]=grp.size(); f["los_hours"]=grp["bloc"].max()*4
    f["mechvent_ever"]=grp["mechvent"].max(); f["vaso_max"]=grp["max_dose_vaso"].max()
    f["fluid_balance_last"]=grp["cumulated_balance"].last(); f["urine_total"]=grp["output_step"].sum()
    for c in tv:
        f[f"{c}_mean"]=grp[c].mean(); f[f"{c}_min"]=grp[c].min()
        f[f"{c}_max"]=grp[c].max();  f[f"{c}_last"]=grp[c].last()
    for c in ["Creatinine","Arterial_lactate","SOFA","Shock_Index","GCS"]:
        f[f"{c}_delta"]=grp[c].last()-grp[c].first()
    X = pd.DataFrame(f); X["morta_90"]=grp["morta_90"].max(); X["died_in_hosp"]=grp["died_in_hosp"].max()
    return X.reset_index()

def load_patients():
    p = _find("sepsis_patients.csv")
    if p:
        print(f"✓ loaded {p}"); return pd.read_csv(p)
    print("sepsis_patients.csv not found — building it from the time-series file…")
    return build_patient_table(load_csv("sepsis_timeseries.csv"))

In [3]:
ts = load_csv("sepsis_timeseries.csv")

# ---- the locked split: by patient, fixed seed, identical for everybody --------
from sklearn.model_selection import train_test_split

stay_outcome = ts.groupby("icustayid")["morta_90"].max()
TRAIN_IDS, TEST_IDS = train_test_split(
    stay_outcome.index.to_numpy(), test_size=0.25,
    stratify=stay_outcome.to_numpy(), random_state=2026)
TRAIN_IDS, TEST_IDS = set(TRAIN_IDS.tolist()), set(TEST_IDS.tolist())

ts_train = ts[ts["icustayid"].isin(TRAIN_IDS)].copy()
ts_test  = ts[ts["icustayid"].isin(TEST_IDS)].copy()

# The held-out labels — you may look, but using them to fit is cheating on yourself. 🙂
Y_TEST = ts_test.groupby("icustayid")["morta_90"].max()

print(f"training stays : {len(TRAIN_IDS):,}  ({ts_train.shape[0]:,} bloc-rows)")
print(f"held-out stays : {len(TEST_IDS):,}  ({ts_test.shape[0]:,} bloc-rows)")
print(f"mortality — train {stay_outcome[list(TRAIN_IDS)].mean():.1%} · "
      f"test {Y_TEST.mean():.1%}   (stratified, so these should match)")
assert not (TRAIN_IDS & TEST_IDS), "patient overlap!"

✓ loaded data/sepsis_timeseries.csv


training stays : 1,272  (27,995 bloc-rows)
held-out stays : 424  (9,709 bloc-rows)
mortality — train 18.2% · test 18.2%   (stratified, so these should match)


## 🧮 The scorer & the leaderboard

`score(preds, name)` takes a **Series of predicted probabilities indexed by `icustayid`** covering
every held-out stay, and reports the metrics that matter clinically:

- **ROC-AUC** — ranking quality (the headline number)
- **Average precision** — the honest metric under class imbalance
- **Brier score** — calibration + accuracy combined (**lower is better**)
- **Sensitivity @ 10% alert rate** — "if the ICU can chase 10% of patients, how many deaths do we catch?"

…and, crucially, a **bootstrap 95% confidence interval on the AUC**. With ~420 held-out patients that
interval is wide. If your lead over the baseline is inside it, you have not actually won — you have
observed noise. (This is Notebook 13's lesson, applied to yourself.)

It also refuses obviously invalid submissions.

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

LEADERBOARD = []
_rng = np.random.default_rng(RANDOM_STATE)

def _auc_ci(y, p, n_boot=400):
    """Bootstrap 95% CI for the AUC — how much of your lead is real?"""
    y, p = np.asarray(y), np.asarray(p)
    aucs = []
    for _ in range(n_boot):
        i = _rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) > 1:
            aucs.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

def score(preds, name="my model", verbose=True):
    """Score a submission. `preds` = Series of P(death) indexed by icustayid."""
    preds = pd.Series(preds).astype(float)
    missing = set(Y_TEST.index) - set(preds.index)
    if missing:
        raise ValueError(f"{len(missing)} held-out stays have no prediction (e.g. {list(missing)[:3]})")
    if preds.isna().any():
        raise ValueError("submission contains NaNs")
    if preds.min() < 0 or preds.max() > 1:
        raise ValueError("predictions must be probabilities in [0, 1]")
    p = preds.reindex(Y_TEST.index).clip(1e-6, 1 - 1e-6)

    k = max(1, int(round(0.10 * len(p))))              # top 10% highest-risk
    flagged = p.nlargest(k).index
    sens_at_10 = Y_TEST.loc[flagged].sum() / max(1, Y_TEST.sum())
    lo, hi = _auc_ci(Y_TEST, p)

    res = {"name": name,
           "ROC_AUC": roc_auc_score(Y_TEST, p),
           "AUC_lo": lo, "AUC_hi": hi,
           "avg_precision": average_precision_score(Y_TEST, p),
           "brier": brier_score_loss(Y_TEST, p),
           "sens_at_10pct_alerts": float(sens_at_10)}
    LEADERBOARD.append(res)
    if verbose:
        print(f"📊 {name}")
        print(f"   ROC-AUC ............... {res['ROC_AUC']:.4f}   [95% CI {lo:.3f}–{hi:.3f}]")
        print(f"   Average precision ..... {res['avg_precision']:.4f}   (base rate {Y_TEST.mean():.3f})")
        print(f"   Brier (lower better) .. {res['brier']:.4f}")
        print(f"   Sensitivity @10% alerts {res['sens_at_10pct_alerts']:.1%}")
        if res["ROC_AUC"] > 0.93:
            print("   🚨 That is suspiciously good. Check for leakage before you celebrate (NB08).")
    return res

def leaderboard():
    return (pd.DataFrame(LEADERBOARD)
              .sort_values("ROC_AUC", ascending=False)
              .reset_index(drop=True).round(4))

def board_score(name):
    """Look a previous submission back up by name."""
    return next(r for r in reversed(LEADERBOARD) if r["name"] == name)

## 🥉 Benchmark 0 — the null model

Always know what "doing nothing" scores. Predict the base rate for everybody.

In [5]:
score(pd.Series(Y_TEST.mean(), index=Y_TEST.index), "null model (base rate)");

📊 null model (base rate)
   ROC-AUC ............... 0.5000   [95% CI 0.500–0.500]
   Average precision ..... 0.1816   (base rate 0.182)
   Brier (lower better) .. 0.1486
   Sensitivity @10% alerts 13.0%


## 🥈 Benchmark 1 — a single clinical variable

Before any machine learning: how far does **worst SOFA during the stay** get you on its own?
This is the bar a real clinical reviewer will hold you to — *"is your model better than the score
we already use?"*

In [6]:
sofa_max_test = ts_test.groupby("icustayid")["SOFA"].max()
score(sofa_max_test / sofa_max_test.max(), "worst SOFA alone");

📊 worst SOFA alone
   ROC-AUC ............... 0.6768   [95% CI 0.609–0.750]
   Average precision ..... 0.3519   (base rate 0.182)
   Brier (lower better) .. 0.2022
   Sensitivity @10% alerts 24.7%


## 🥇 The starter baseline — beat this

The Notebook 05 recipe, unchanged: aggregate to one row per stay, impute + scale inside a pipeline,
gradient boosting. Everything is fit on **training stays only**.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

def features_from(ts_slice):
    """One row per stay. Uses only that stay's own data — no cross-patient info."""
    tab = build_patient_table(ts_slice).set_index("icustayid")
    return tab.drop(columns=["morta_90", "died_in_hosp"])

Xtr = features_from(ts_train)
Xte = features_from(ts_test).reindex(columns=Xtr.columns)   # same columns, same order
ytr = ts_train.groupby("icustayid")["morta_90"].max().reindex(Xtr.index)

baseline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                          scale_pos_weight=float((ytr == 0).sum() / max(1, (ytr == 1).sum())),
                          random_state=RANDOM_STATE, n_jobs=-1)),
])
baseline.fit(Xtr, ytr)

pred_baseline = pd.Series(baseline.predict_proba(Xte)[:, 1], index=Xte.index)
score(pred_baseline, "starter baseline (XGBoost)");

📊 starter baseline (XGBoost)
   ROC-AUC ............... 0.7540   [95% CI 0.696–0.800]
   Average precision ..... 0.3966   (base rate 0.182)
   Brier (lower better) .. 0.1462
   Sensitivity @10% alerts 24.7%


In [8]:
leaderboard()

,name,ROC_AUC,AUC_lo,AUC_hi,avg_precision,brier,sens_at_10pct_alerts
0,starter baseline (XGBoost),0.7540,0.6958,0.7998,0.3966,0.1462,0.2468
1,worst SOFA alone,0.6768,0.6093,0.7497,0.3519,0.2022,0.2468
2,null model (base rate),0.5000,0.5000,0.5000,0.1816,0.1486,0.1299


## 🧪 Sanity check before you optimise: cross-validate on the *training* stays

The right way to compare your ideas is **CV on the training set** — not repeated submissions.
Every time you look at the held-out score and change something, you leak a little information into
your model choice. Do that thirty times and your test score is fiction.

Use this as your working loop; submit maybe two or three times all session.

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(baseline, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"5-fold CV ROC-AUC on training stays: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
bl = board_score("starter baseline (XGBoost)")
print(f"Held-out ROC-AUC of the same model : {bl['ROC_AUC']:.4f}  "
      f"[95% CI {bl['AUC_lo']:.3f}-{bl['AUC_hi']:.3f}]")
print("\nIf these two disagree a lot, your CV isn't measuring what you think it is.")

5-fold CV ROC-AUC on training stays: 0.7788 ± 0.0485
Held-out ROC-AUC of the same model : 0.7540  [95% CI 0.696-0.800]

If these two disagree a lot, your CV isn't measuring what you think it is.


## 💡 Your turn — the idea board

Ranked roughly by **how much they usually help**, which is *not* the order people try them in:

**1. Better features (biggest win — Notebook 04)**
- **Trajectory beats snapshot.** `last − first`, slope over the last 24 h (6 blocs), max rate of change.
  A *rising* lactate is worse than a high but falling one; a single `max()` cannot see that.
- **Early-window features**: restrict to the first 24 h (`bloc <= 6`) and see how much you lose —
  that is the honest early-warning model, and it's the one a hospital would actually deploy.
- **Treatment intensity**: cumulative vasopressor dose, hours ventilated, fluid balance trajectory.
- **Missingness as a feature**: `df[col].isna().mean()` per stay. Whether a lab was *ordered* carries
  information about how worried the team was.
- **Variability**: `std()` of HR / MAP / lactate. Instability is a signal that means never capture.

**2. Cleaner data (Notebook 02, measured honestly in Notebook 09)**
- Fix the Fahrenheit/Celsius mess, winsorise the labs, forward-fill *within* a stay.

**3. The right model family**
- XGBoost / LightGBM / HistGradientBoosting are strong on tabular ICU data. Logistic regression on
  well-engineered features is a very respectable, and much more explainable, competitor.
- **Ensemble**: average the probabilities of 2–3 different families. Cheap, almost always helps.

**4. Calibration (helps Brier a lot, AUC not at all)**
- `CalibratedClassifierCV(..., method="isotonic", cv=5)` around your model.

**5. Hyperparameters (last, and worth the least — Notebook 09)**
- `RandomizedSearchCV` with a budget, scored by CV. Expect +0.005, not +0.05.

In [10]:
# ============================================================================
#  ✏️  YOUR MODEL — edit freely.
#  Build features from ts_train, fit on training stays only, predict for Xte.
# ============================================================================

def my_features(ts_slice):
    """Start from the baseline table and add your own ideas."""
    tab = build_patient_table(ts_slice).set_index("icustayid")
    tab = tab.drop(columns=["morta_90", "died_in_hosp"])
    g = ts_slice.sort_values(["icustayid", "bloc"]).groupby("icustayid")

    # --- example add-on 1: variability of key vitals --------------------------
    for c in ["HR", "MeanBP", "Arterial_lactate", "SOFA"]:
        tab[f"{c}_std"] = g[c].std()

    # --- example add-on 2: trend over the LAST 24 h (6 blocs) -----------------
    last24 = ts_slice[ts_slice["bloc"] > ts_slice.groupby("icustayid")["bloc"].transform("max") - 6]
    g24 = last24.groupby("icustayid")
    for c in ["Arterial_lactate", "SOFA", "Creatinine"]:
        tab[f"{c}_slope24"] = g24[c].last() - g24[c].first()

    # --- example add-on 3: missingness as signal ------------------------------
    for c in ["Arterial_lactate", "Albumin", "INR"]:
        tab[f"{c}_missing_frac"] = g[c].apply(lambda s: s.isna().mean())

    # 👉 your ideas here …

    return tab

Xtr2 = my_features(ts_train)
Xte2 = my_features(ts_test).reindex(columns=Xtr2.columns)

my_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(n_estimators=600, max_depth=4, learning_rate=0.04,
                          subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                          eval_metric="logloss",
                          scale_pos_weight=float((ytr == 0).sum() / max(1, (ytr == 1).sum())),
                          random_state=RANDOM_STATE, n_jobs=-1)),
])

# --- always CV first ---------------------------------------------------------
cv2 = cross_val_score(my_model, Xtr2, ytr, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"CV ROC-AUC: {cv2.mean():.4f} ± {cv2.std():.4f}   (baseline CV was {cv_auc.mean():.4f})")

CV ROC-AUC: 0.7807 ± 0.0507   (baseline CV was 0.7788)


In [11]:
# --- when the CV says you improved, THEN submit ------------------------------
my_model.fit(Xtr2, ytr)
score(pd.Series(my_model.predict_proba(Xte2)[:, 1], index=Xte2.index), "my model v1");
leaderboard()

📊 my model v1
   ROC-AUC ............... 0.7548   [95% CI 0.695–0.810]
   Average precision ..... 0.3962   (base rate 0.182)
   Brier (lower better) .. 0.1476
   Sensitivity @10% alerts 27.3%


,name,ROC_AUC,AUC_lo,AUC_hi,avg_precision,brier,sens_at_10pct_alerts
0,my model v1,0.7548,0.6951,0.8103,0.3962,0.1476,0.2727
1,starter baseline (XGBoost),0.7540,0.6958,0.7998,0.3966,0.1462,0.2468
2,worst SOFA alone,0.6768,0.6093,0.7497,0.3519,0.2022,0.2468
3,null model (base rate),0.5000,0.5000,0.5000,0.1816,0.1486,0.1299


## 🤝 Team leaderboard

Paste your neighbours' numbers in and see where you land — or read yours out to the room.

In [12]:
# LEADERBOARD.append({"name": "Anna & Ben", "ROC_AUC": 0.79, "AUC_lo": 0.74, "AUC_hi": 0.84,
#                     "avg_precision": 0.44, "brier": 0.132, "sens_at_10pct_alerts": 0.30})
leaderboard()

,name,ROC_AUC,AUC_lo,AUC_hi,avg_precision,brier,sens_at_10pct_alerts
0,my model v1,0.7548,0.6951,0.8103,0.3962,0.1476,0.2727
1,starter baseline (XGBoost),0.7540,0.6958,0.7998,0.3966,0.1462,0.2468
2,worst SOFA alone,0.6768,0.6093,0.7497,0.3519,0.2022,0.2468
3,null model (base rate),0.5000,0.5000,0.5000,0.1816,0.1486,0.1299


## 🧠 The debrief — the part that actually matters

Whoever won, spend the last 10 minutes on these. They are the questions a journal reviewer, an
ethics board and a clinical director will ask, in that order.

1. **Would your model still work in another hospital?** Different case-mix, different lab assays,
   different admission thresholds. Expect a drop — Notebook 08's external-validation section.
2. **What would it change?** A risk score nobody acts on is a paper, not an intervention. What is the
   action at the alert, who does it, and what is the alert budget per shift?
3. **What did the winning feature actually encode?** Run SHAP (Notebook 07) on the top model. If the
   biggest driver is something like "an arterial line was placed", the model may have learned
   *clinical concern*, not physiology — a proxy that will vanish the moment practice changes.
4. **Is it fair?** Run the winner through Notebook 13's subgroup report before you believe any of it.
5. **What's the calibration?** A model with better AUC but worse Brier is often the *worse* deployment
   candidate.

> 🩺 **The honest summary of the day:** getting from 0.78 to 0.83 took an hour of clever features.
> Getting from 0.83 to *a system that helps a patient* takes prospective validation, a workflow, and
> a team. The first part is the fun part. The second part is the job.

## ✏️ Extension challenges

- **Early-warning mode.** Rebuild the features using only the **first 24 hours** of each stay
  (`ts[ts.bloc <= 6]`) and re-score. How much AUC do you lose? That gap is the price of usefulness.
- **Per-bloc prediction.** Instead of one prediction per stay, predict at *every* bloc using only
  data up to that point. Score it as a time-dependent AUC. This is what a real bedside model does.
- **Beat XGBoost with logistic regression.** Same features, `LogisticRegression` with splines or
  interaction terms. If you get within 0.01, argue for deploying the explainable one.
- **Best-calibrated model wins.** Re-run the leaderboard sorted by **Brier** instead of AUC. Does the
  winner change? Discuss which competition you'd rather have entered.